# ASR model comparison on CPU (faster-whisper)
- Goal: find out how fast (RTF) and how accurate (WER) different Whisper models are on this laptop, in English and French.
- All models run with faster-whisper (CTranslate2) on the CPU, and all of them get exactly the same audio.
- The test with transformers (FP32) is in `whisper-v3-turbo.ipynb`.

**Metrics**
- RTF = processing time / audio length. If it is below 1, the model is faster than real time. Live transcription needs this.
- WER = (wrong words + missing words + extra words) / number of words in the correct text. We count all clips together, not one by one.

In [1]:
import io
import subprocess
import sys
import time
from pathlib import Path

import jiwer
import pandas as pd
from datasets import Audio, load_dataset
from faster_whisper import WhisperModel, decode_audio
from transformers import WhisperTokenizer

N_SAMPLES = 20                 # per language
CPU_THREADS = 8                # physical cores of the Ryzen 7 PRO 5850U
BEAM_SIZE = 5                  # same as the transformers default
MODELS_DIR = Path("../models")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

C:\Users\Unal\Desktop\DeepLearningAndOptimization\HES-Hackathon\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Data: FLEURS validation set
- We take the first `N_SAMPLES` clips of the English (`en_us`) and French (`fr_fr`) validation sets. Only the validation file is downloaded.
- The audio is decoded one time with PyAV (`decode_audio`), so we do not need `torchcodec` or FFmpeg here.
- The correct text is `raw_transcription` (with capital letters and punctuation).

In [2]:
FLEURS_CONFIGS = {"en": "en_us", "fr": "fr_fr"}

def load_fleurs(lang_config, n=N_SAMPLES):
    url = f"hf://datasets/google/fleurs/parquet-data/{lang_config}/validation-00000-of-00001.parquet"
    ds = load_dataset("parquet", data_files=url, split="train").select(range(n))
    ds = ds.cast_column("audio", Audio(decode=False))   # raw bytes, decoded below
    samples = []
    for ex in ds:
        audio = decode_audio(io.BytesIO(ex["audio"]["bytes"]), sampling_rate=16000)
        samples.append({"id": ex["id"], "audio": audio,
                        "duration_s": len(audio) / 16000,
                        "reference": ex["raw_transcription"]})
    return samples

data = {lang: load_fleurs(cfg) for lang, cfg in FLEURS_CONFIGS.items()}
{lang: f"{len(s)} samples, {sum(x['duration_s'] for x in s):.0f} s of audio" for lang, s in data.items()}

{'en': '20 samples, 173 s of audio', 'fr': '20 samples, 189 s of audio'}

## 2. Convert large-v3-turbo to CTranslate2
- faster-whisper cannot read the original `model.safetensors` file. The converter writes the same weights again in the CTranslate2 format.
- It uses the official `openai/whisper-large-v3-turbo` model that is already in the Hugging Face cache, so nothing new is downloaded.
- The file is saved in FP16. Nothing is lost, because the original weights are also FP16. We choose the precision for running the model later with `compute_type`.
- This runs only once. If the folder already exists, the cell skips it.

In [3]:
TURBO_DIR = MODELS_DIR / "whisper-large-v3-turbo-ct2"

if not (TURBO_DIR / "model.bin").exists():
    converter = Path(sys.executable).parent / "ct2-transformers-converter"
    subprocess.run([str(converter),
                    "--model", "openai/whisper-large-v3-turbo",
                    "--output_dir", str(TURBO_DIR),
                    "--copy_files", "tokenizer.json", "preprocessor_config.json",
                    "--quantization", "float16"], check=True)

sorted(p.name for p in TURBO_DIR.iterdir())

['config.json',
 'model.bin',
 'preprocessor_config.json',
 'tokenizer.json',
 'vocabulary.json']

## 3. Benchmark function
- It loads the model and measures the load time separately. Then it does one warm-up run, and after that it measures the time for every clip.
- The language is fixed (`task="transcribe"`), so a wrong language guess cannot change the results.
- `vad_filter` is off: each FLEURS clip is one read sentence, and the transformers test does not use VAD either.
- The results are saved as CSV files in `results/`. If a CSV file already exists, the function reads it and does not run the model again.
  To measure again, delete the CSV file.

In [4]:
def benchmark(label, model_path, compute_type):
    csv = RESULTS_DIR / f"fleurs_{label}_{compute_type}.csv"
    if csv.exists():                      # already measured: reuse instead of re-running
        return pd.read_csv(csv)

    t0 = time.perf_counter()
    model = WhisperModel(model_path, device="cpu", compute_type=compute_type, cpu_threads=CPU_THREADS)
    load_s = time.perf_counter() - t0

    def transcribe(audio, lang):
        segments, _ = model.transcribe(audio, language=lang, task="transcribe",
                                       beam_size=BEAM_SIZE, vad_filter=False)
        return " ".join(s.text.strip() for s in segments)   # the generator does the work

    transcribe(data["en"][0]["audio"], "en")   # warm-up, not measured

    rows = []
    for lang, samples in data.items():
        for ex in samples:
            t0 = time.perf_counter()
            hyp = transcribe(ex["audio"], lang)
            rows.append({"model": label, "compute_type": compute_type, "lang": lang, "id": ex["id"],
                         "duration_s": ex["duration_s"], "time_s": time.perf_counter() - t0,
                         "reference": ex["reference"], "hypothesis": hyp})
        print(f"{label} {compute_type} {lang}: done")

    df = pd.DataFrame(rows).assign(load_s=load_s)
    df.to_csv(csv, index=False)
    del model
    return df

## 4. Run the models
| Run | Why |
| --- | --- |
| `small` int8 | Small and fast model, used as a baseline |
| `large-v3-turbo` int8 | Main candidate |
| `large-v3-turbo` float32 | Same model without int8: shows what int8 changes, and lets us compare with transformers FP32 |

In [5]:
RUNS = [
    ("small", "small", "int8"),
    ("large-v3-turbo", str(TURBO_DIR), "int8"),
    ("large-v3-turbo", str(TURBO_DIR), "float32"),
]

results = pd.concat([benchmark(*run) for run in RUNS], ignore_index=True)

## 5. WER and RTF
- Before we compute WER, we clean both texts in the same way ("normalise"). English uses Whisper's English normaliser (lowercase, no punctuation, same spelling and number format). French uses the basic normaliser (lowercase, no punctuation, accents are kept).
- We remove brackets before normalising. If we do not, the normaliser deletes the words inside the brackets from the correct text.

In [6]:
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-large-v3-turbo")

def normalize(text, lang):
    # Whisper's normalisers delete text inside (...) and [...]; FLEURS references use parentheses
    # for spoken words, so only the brackets themselves are removed
    text = text.translate(str.maketrans("()[]", "    "))
    return tokenizer.normalize(text) if lang == "en" else tokenizer.basic_normalize(text)

results["ref_norm"] = [normalize(t, l) for t, l in zip(results["reference"], results["lang"])]
results["hyp_norm"] = [normalize(t, l) for t, l in zip(results["hypothesis"], results["lang"])]

def summarize(g):
    return pd.Series({
        "WER %": 100 * jiwer.wer(list(g["ref_norm"]), list(g["hyp_norm"])),
        "RTF": g["time_s"].sum() / g["duration_s"].sum(),
        "audio s": g["duration_s"].sum(),
        "load s": g["load_s"].iloc[0],
    })

summary = results.groupby(["model", "compute_type", "lang"]).apply(summarize).round(3)
summary

WER %    RTF  audio s  load s
model          compute_type lang                                
large-v3-turbo float32      en     4.188  3.016   173.38   4.855
                            fr     4.497  2.636   188.70   4.855
               int8         en     4.188  1.908   173.38  11.028
                            fr     3.640  1.717   188.70  11.028
small          int8         en     6.021  0.577   173.38   3.112
                            fr    10.707  0.594   188.70   3.112

## 6. RTF by clip length
- Whisper always fills the audio up to 30 seconds. So the encoder does the same work for a 4-second clip and for a 20-second clip.
- Live transcription will work with short clips, so the RTF of short clips is the most important.

In [7]:
results["rtf"] = results["time_s"] / results["duration_s"]
results["length"] = pd.cut(results["duration_s"], [0, 6, 12, 60], labels=["< 6 s", "6-12 s", "> 12 s"])
results.pivot_table(index=["model", "compute_type"], columns="length", values="rtf",
                    aggfunc="median", observed=True).round(2)

length                       < 6 s  6-12 s  > 12 s
model          compute_type                       
large-v3-turbo float32        5.96    2.99    1.82
               int8           3.60    2.00    1.15
small          int8           0.94    0.59    0.47

## 7. Error check
- The sentences with the most errors for each model. Errors in numbers and names are the most important for fact-checking.

In [8]:
results["errors"] = [jiwer.process_words(r, h).wer for r, h in zip(results["ref_norm"], results["hyp_norm"])]

pd.set_option("display.max_colwidth", None)
(results.sort_values("errors", ascending=False)
        .groupby(["model", "compute_type"]).head(3)
        [["model", "compute_type", "lang", "errors", "ref_norm", "hyp_norm"]])

,model,compute_type,lang,errors,ref_norm,hyp_norm
3,small,int8,en,0.600000,then lakkha singh took the lead in singing the bhajans,then like a sing to the lead and singing the basjams
43,large-v3-turbo,int8,en,0.500000,then lakkha singh took the lead in singing the bhajans,then laka sing to the lead and singing the bashan
83,large-v3-turbo,float32,en,0.500000,then lakkha singh took the lead in singing the bhajans,then laka sing to the lead and singing the bashan
31,small,int8,fr,0.333333,sur les routes glacées et enneigées le frottement est faible et vous ne pouvez pas conduire comme si vous étiez sur l asphalte nu,sur d un coup de glacé et en neige le frottement est faible et vous ne pouvez pas conduire même si vous étiez sur l asphalte nu
21,small,int8,fr,0.291667,il n avait pas non plus le pouvoir de déroger aux lois fiscales ni à celles sur les droits de douane entre les états,il n avait pas non plus le pouvoir de déroger au loi fiscale il y a 7 sur les droits de douane entre les états
63,large-v3-turbo,int8,fr,0.173913,de nombreux formats courants famille de formats aps par exemple sont égaux à ce rapport d aspect ou s en approchent de près,de nombreux formats courants familles de formats aps par exemple sont égaux à ce rapport d aspect ou sans rapproche de près
103,large-v3-turbo,float32,fr,0.173913,de nombreux formats courants famille de formats aps par exemple sont égaux à ce rapport d aspect ou s en approchent de près,de nombreux formats courants familles de formats aps par exemple sont égaux à ce rapport d aspect ou sans rapproche de près
101,large-v3-turbo,float32,fr,0.166667,il n avait pas non plus le pouvoir de déroger aux lois fiscales ni à celles sur les droits de douane entre les états,il n avait pas non plus le pouvoir de déroger aux lois fiscales il y a celle sur les droits de douane entre les états
41,large-v3-turbo,int8,en,0.151515,now widely available throughout the archipelago javanese cuisine features an array of simply seasoned dishes the predominant flavorings the javanese favor being peanuts chillies sugar especially javanese coconut sugar and various aromatic spices,now widely available throughout the archipelago japanese cuisines features an array of simply seasoned dishes the predominant flavorings the japanese favor being peanuts chilies sugar especially japanese coconut sugar and various aromatic spices


## 8. Findings
Measured on 2026-10-20, AMD Ryzen 7 PRO 5850U (8 cores), 32 GB RAM, CPU only, 20 FLEURS validation clips per language (382 words in English, 467 in French).

| Model | Engine | Precision | WER en | WER fr | RTF en | RTF fr | RTF clips < 6 s | Load |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| small | CTranslate2 | int8 | 6.0 % | 10.7 % | 0.58 | 0.59 | 0.94 | 3 s |
| large-v3-turbo | CTranslate2 | int8 | 4.2 % | 3.6 % | 1.91 | 1.72 | 3.60 | 11 s |
| large-v3-turbo | CTranslate2 | float32 | 4.2 % | 4.5 % | 3.02 | 2.64 | 5.96 | 5 s |
| large-v3-turbo | transformers | float32 | 4.2 % | 3.9 % | 3.40 | 3.16 | 6.11 | - |

The transformers row comes from `whisper-v3-turbo.ipynb` (same clips, WER computed in the same way). Its load time was not measured.

- **Only `small` is faster than real time.** Turbo int8 is about 1.8 times slower than real time, and about 3.6 times slower on short clips.
- **Turbo is much better in French** (3.6 % vs 10.7 %). In English the difference is smaller (4.2 % vs 6.0 %).
- **int8 makes turbo about 1.6 times faster, and WER does not get worse.** In French, int8 and float32 differ by only 4 words (17 vs 21 errors in 467 words). With 20 clips, this is just noise.
- **Short clips are the main problem:** every call pays for a full 30-second window, so RTF gets worse when clips get shorter. Live transcription works with short clips.
- int8 takes longer to load (11 s vs 5 s), because the FP16 weights are converted to int8 when the model is loaded.
- With the same precision (float32), CTranslate2 is only a little faster than transformers (RTF 3.02 vs 3.40 in English). On this CPU, most of the speed-up comes from int8, not from the engine.

**Limits:** 20 clips per language is a small test. Differences smaller than about 2 WER points are not reliable. FLEURS is clean, read speech, so WER on real videos will be higher.